In [ ]:
import json
import random
import re
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

REPO_ROOT = Path('/Users/itaishapira/Desktop/knowledge_gap/LLMsKnow').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sycophancy_bias_probe.correctness import extract_gold_answers_from_base, grade_short_answer, normalize_answer
from sycophancy_bias_probe.dataset import build_question_groups, deduplicate_rows, split_groups_train_val_test, template_type as infer_template_type

def read_jsonl(path: str):
    rows = []
    with open(path, 'r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 240)
sns.set_style('white')

PALETTE = {
    'neutral': '#73b3ab',
    'biased': '#d4651a',
    'incorrect_suggestion': '#d4651a',
    'doubt_correct': '#73b3ab',
    'suggest_correct': '#8d99ae',
    'ambiguous': '#bcb8b1',
    'usable': '#264653',
}
PROBE_PALETTE = {
    'probe_no_bias': '#264653',
    'probe_bias_incorrect_suggestion': '#d4651a',
    'probe_bias_doubt_correct': '#73b3ab',
    'probe_bias_suggest_correct': '#8d99ae',
}
TEMPLATE_SEED_OFFSET = {
    'neutral': 11,
    'incorrect_suggestion': 22,
    'doubt_correct': 33,
    'suggest_correct': 44,
}
AUDIT_SEED = 7
USE_MODEL_TOKENIZER = False
RUN_DIR_INPUT = Path('/Users/itaishapira/Desktop/knowledge_gap/LLMsKnow/output/sycophancy_bias_probe/mistralai_Mistral_7B_Instruct_v0_2/fast_dirty_mistral7b_instruct_V2')

def resolve_run_dir(run_dir_input: Path) -> Path:
    candidates = [Path(run_dir_input), Path(run_dir_input) / Path(run_dir_input).name]
    for candidate in candidates:
        if (candidate / 'sampled_responses.csv').exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find a run directory with sampled_responses.csv under {run_dir_input}')

def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def parse_json_list(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return [value]
        return parsed if isinstance(parsed, list) else [parsed]
    return [value]

def show_question(index: int, text: str):
    display(Markdown(f'### {index}) {text}'))

def show_answer(text: str):
    display(Markdown(f'**Answer.** {text}'))

def finish_ax(ax, title: str, xlabel: str, ylabel: str, *, legend: bool = False, ncol: int = 1):
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.tick_params(axis='both', labelsize=12)
    if legend and ax.get_legend() is not None:
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=ncol, frameon=False)
    sns.despine(ax=ax)

def question_type_bucket(question: str) -> str:
    text = str(question or '').strip()
    if not text:
        return 'unknown'
    token = re.sub(r'[^A-Za-z]', '', text.split()[0]).lower()
    return token or 'unknown'

def answer_format_bucket(text: str) -> str:
    text = str(text or '').strip()
    if not text:
        return 'empty'
    if '\n' in text:
        return 'multiline'
    if re.search(r'\bor\b|/', text, flags=re.IGNORECASE):
        return 'multi_candidate_like'
    n_words = len(text.split())
    if n_words == 1:
        return 'single_token'
    if n_words <= 4:
        return 'short_phrase'
    return 'sentence_or_long'

def template_to_probe_name(template_type: str) -> str:
    return 'probe_no_bias' if template_type == 'neutral' else f'probe_bias_{template_type}'

def id_tuples(df: pd.DataFrame, cols):
    return set(map(tuple, df.loc[:, list(cols)].drop_duplicates().itertuples(index=False, name=None)))

def jaccard(a, b) -> float:
    union = a | b
    return len(a & b) / len(union) if union else np.nan

@lru_cache(maxsize=1)
def get_tokenizer():
    if not USE_MODEL_TOKENIZER:
        return None
    try:
        from transformers import AutoTokenizer
        cache_dir = (CTX['run_config'] or {}).get('hf_cache_dir')
        return AutoTokenizer.from_pretrained(CTX['model_name'], use_fast=True, cache_dir=cache_dir)
    except Exception as exc:
        print(f'Tokenizer unavailable, falling back to whitespace token counts. ({type(exc).__name__}: {exc})')
        return None

def token_count_mode() -> str:
    return 'model-tokenized' if USE_MODEL_TOKENIZER and get_tokenizer() is not None else 'approximate_whitespace'

def count_tokens(text: str) -> int:
    tokenizer = get_tokenizer()
    text = str(text or '')
    if tokenizer is None:
        return len(text.split())
    return len(tokenizer(text, add_special_tokens=False).input_ids)

def looks_truncated(text: str, token_count: int, max_new_tokens: int) -> bool:
    tail = str(text or '').rstrip()
    if not tail:
        return False
    near_ceiling = token_count >= max(1, max_new_tokens - 2)
    clean_stop_chars = {'.', '!', '?', '"', "'", ')', ']', '}'}
    return near_ceiling and tail[-1] not in clean_stop_chars

def load_context():
    run_dir = resolve_run_dir(RUN_DIR_INPUT)
    run_config = load_json(run_dir / 'run_config.json') or {}
    sampled = pd.read_csv(run_dir / 'sampled_responses.csv')
    tuples = pd.read_csv(run_dir / 'final_tuples.csv')
    summary = pd.read_csv(run_dir / 'summary_by_question.csv')
    probe_metadata = load_json(run_dir / 'probe_metadata.json') or {}
    sampling_manifest = load_json(run_dir / 'sampling_manifest.json') or {}
    status = load_json(run_dir / 'status.json') or {}

    sampled['gold_answers_list'] = sampled['gold_answers'].map(parse_json_list)
    tuples['gold_answers_list'] = tuples['gold_answers'].map(parse_json_list)

    bias_types = run_config.get('bias_types', [])
    if isinstance(bias_types, str):
        bias_types = [x.strip() for x in bias_types.split(',') if x.strip()]
    template_order = ['neutral', *bias_types]

    data_dir = REPO_ROOT / run_config.get('data_dir', 'data/sycophancy-eval')
    input_path = data_dir / run_config.get('input_jsonl', 'answer.jsonl')
    raw_rows = read_jsonl(str(input_path))
    raw_rows_dedup = deduplicate_rows(raw_rows)
    groups_all = build_question_groups(raw_rows_dedup, selected_bias_types=bias_types)

    groups_for_run = list(groups_all)
    max_questions = run_config.get('max_questions')
    if max_questions is not None:
        rng = random.Random(int(run_config.get('split_seed', 0)))
        rng.shuffle(groups_for_run)
        groups_for_run = groups_for_run[: int(max_questions)]

    train_groups, val_groups, test_groups = split_groups_train_val_test(
        groups_for_run,
        test_frac=float(run_config.get('test_frac', 0.2)),
        val_frac=float(run_config.get('probe_val_frac', 0.2)),
        seed=int(run_config.get('split_seed', 0)),
    )

    question_level = sampled.groupby(['split', 'question_id'], as_index=False).agg(
        question=('question', 'first'),
        correct_answer=('correct_answer', 'first'),
        incorrect_answer=('incorrect_answer', 'first'),
        n_templates=('template_type', 'nunique'),
        n_records=('record_id', 'count'),
    )
    neutral_acc = sampled[sampled['template_type'] == 'neutral'].groupby('question_id')['correctness'].mean()
    question_level['neutral_acc'] = question_level['question_id'].map(neutral_acc)

    return {
        'run_dir': run_dir,
        'run_config': run_config,
        'sampled': sampled,
        'tuples': tuples,
        'summary': summary,
        'probe_metadata': probe_metadata,
        'sampling_manifest': sampling_manifest,
        'status': status,
        'bias_types': bias_types,
        'template_order': template_order,
        'raw_rows': raw_rows,
        'raw_rows_dedup': raw_rows_dedup,
        'groups_all': groups_all,
        'groups_for_run': groups_for_run,
        'train_groups': train_groups,
        'val_groups': val_groups,
        'test_groups': test_groups,
        'question_level': question_level,
        'model_name': run_config.get('model') or sampled['model_name'].iloc[0],
    }

CTX = load_context()
sampled = CTX['sampled']
tuples = CTX['tuples']
summary = CTX['summary']

show_question(1, 'Which questions should I sample to make sure the dataset looks sensible, diverse, and correctly paired with correct_answer and incorrect_answer?')

question_level = CTX['question_level'].copy()
question_level['question_len'] = question_level['question'].str.len()
question_level['correct_len'] = question_level['correct_answer'].str.len()
question_level['incorrect_len'] = question_level['incorrect_answer'].str.len()
question_level['question_type'] = question_level['question'].map(question_type_bucket)

candidate_frames = [
    question_level.sort_values('question_len').head(1),
    question_level.sort_values('question_len', ascending=False).head(1),
    question_level.sort_values('correct_len', ascending=False).head(1),
    question_level.sort_values('incorrect_len', ascending=False).head(1),
]
for split_name in ['train', 'val', 'test']:
    split_df = question_level[question_level['split'] == split_name]
    if len(split_df):
        candidate_frames.append(split_df.sample(1, random_state=AUDIT_SEED + len(candidate_frames)))

recommended = pd.concat(candidate_frames, ignore_index=True).drop_duplicates('question_id')
remaining = question_level[~question_level['question_id'].isin(recommended['question_id'])]
if len(recommended) < min(8, len(question_level)) and len(remaining):
    top_up = remaining.sample(min(8 - len(recommended), len(remaining)), random_state=AUDIT_SEED + 99)
    recommended = pd.concat([recommended, top_up], ignore_index=True)

recommended = recommended[
    ['split', 'question_id', 'question', 'correct_answer', 'incorrect_answer', 'question_type', 'neutral_acc', 'n_templates', 'n_records']
].sort_values(['split', 'question_id']).reset_index(drop=True)

show_answer(
    f'Start with {len(recommended)} deterministic audit questions: shortest and longest wording, longest answers, plus at least one question from each split. This mix checks coverage, formatting, and answer pairing before deeper analysis.'
)
display(recommended)


In [ ]:
show_question(2, 'Which prompts should I sample for the same question to confirm that the neutral and biased versions differ only by the intended bias text?')

prompt_df = sampled[
    ['split', 'question_id', 'question', 'template_type', 'prompt_text', 'correct_answer', 'incorrect_answer']
].drop_duplicates().copy()
prompt_df['suffix_after_question'] = prompt_df.apply(
    lambda row: row['prompt_text'][len(row['question']):].strip()
    if str(row['prompt_text']).startswith(str(row['question']))
    else '<question text not found as prompt prefix>',
    axis=1,
)

sampled_question_ids = (
    prompt_df[['split', 'question_id']]
    .drop_duplicates()
    .sample(min(3, prompt_df['question_id'].nunique()), random_state=AUDIT_SEED + 2)['question_id']
    .tolist()
)
prompt_sample = prompt_df[prompt_df['question_id'].isin(sampled_question_ids)].sort_values(['question_id', 'template_type'])

show_answer(
    'Inspect the rows below question by question. The `suffix_after_question` column should contain only the intended bias text, while the base question and the answer fields stay unchanged across prompt variants.'
)
display(prompt_sample.reset_index(drop=True))

suffix_examples = (
    prompt_df.groupby('template_type')['suffix_after_question']
    .agg(lambda values: list(dict.fromkeys(values))[:3])
    .rename('example_suffixes')
    .reset_index()
)
display(suffix_examples)


In [ ]:
show_question(3, 'For one randomly chosen prompt, what do several raw model responses look like, how were they parsed/labeled, and how long are they?')

prompt_pool = sampled[['split', 'question_id', 'template_type', 'prompt_text']].drop_duplicates()
chosen_prompt = prompt_pool.sample(1, random_state=AUDIT_SEED + 3).iloc[0]
chosen_rows = sampled[
    (sampled['split'] == chosen_prompt['split'])
    & (sampled['question_id'] == chosen_prompt['question_id'])
    & (sampled['template_type'] == chosen_prompt['template_type'])
].copy()
chosen_rows['response_tokens'] = chosen_rows['response_raw'].map(count_tokens)
chosen_rows['response_chars'] = chosen_rows['response_raw'].fillna('').str.len()

avg_tokens = chosen_rows['response_tokens'].mean() if len(chosen_rows) else np.nan
show_answer(
    f"Prompt chosen with seed {AUDIT_SEED + 3}. Across {len(chosen_rows)} draws, the average completion length is {avg_tokens:.1f} token-units using `{token_count_mode()}` counting. The table below shows raw text, parsed short answer, grading label, and response length side by side."
)
display(Markdown(f"**Prompt text**\n\n```text\n{chosen_prompt['prompt_text']}\n```"))
display(
    chosen_rows[
        ['draw_idx', 'response_raw', 'response', 'correctness', 'grading_status', 'grading_reason', 'usable_for_metrics', 'response_tokens', 'response_chars']
    ].reset_index(drop=True)
)


In [ ]:
show_question(4, 'Are many responses hitting the length ceiling, and do any look cut off in the middle because of the small max_new_tokens?')

length_df = sampled[['record_id', 'split', 'question_id', 'template_type', 'response_raw']].copy()
length_df['response_tokens'] = length_df['response_raw'].map(count_tokens)
max_new_tokens = int(CTX['run_config'].get('max_new_tokens', 0))
length_df['near_ceiling'] = length_df['response_tokens'] >= max(1, max_new_tokens - 2)
length_df['looks_truncated'] = length_df.apply(
    lambda row: looks_truncated(row['response_raw'], row['response_tokens'], max_new_tokens),
    axis=1,
)

near_ceiling_count = int(length_df['near_ceiling'].sum())
truncated_count = int(length_df['looks_truncated'].sum())
show_answer(
    f'{near_ceiling_count}/{len(length_df)} responses are within two token-units of max_new_tokens={max_new_tokens} using `{token_count_mode()}` counting, and {truncated_count} meet a simple truncation heuristic. Use the histogram and the suspicious examples table below to see whether short generation limits are clipping answers.'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    data=length_df,
    x='response_tokens',
    hue='template_type',
    hue_order=CTX['template_order'],
    palette=PALETTE,
    multiple='stack',
    bins=20,
    ax=ax,
)
ax.axvline(max_new_tokens, color='black', linestyle='--', linewidth=1.5, label=f'max_new_tokens={max_new_tokens}')
finish_ax(ax, 'Completion Length Distribution', 'Completion tokens', 'Count', legend=True, ncol=3)
plt.tight_layout()
plt.show()

suspicious = length_df[length_df['looks_truncated']].sort_values('response_tokens', ascending=False)
display(suspicious[['record_id', 'split', 'question_id', 'template_type', 'response_tokens', 'response_raw']].head(12).reset_index(drop=True))


In [ ]:
show_question(5, 'For a manual sample of labeled responses, do the parsed short answers actually match the raw generations and the assigned labels?')

usable = sampled[sampled['usable_for_metrics']].copy()
manual_parts = []
for template_type in CTX['template_order']:
    for correctness_value in [0.0, 1.0]:
        bucket = usable[(usable['template_type'] == template_type) & (usable['correctness'] == correctness_value)]
        if len(bucket):
            manual_parts.append(
                bucket.sample(
                    min(2, len(bucket)),
                    random_state=AUDIT_SEED + TEMPLATE_SEED_OFFSET.get(template_type, 0) + int(correctness_value),
                )
            )
manual_sample = pd.concat(manual_parts, ignore_index=True).drop_duplicates('record_id')

regraded = manual_sample.apply(lambda row: grade_short_answer(row['response_raw'], row['gold_answers_list']), axis=1)
manual_sample['reparsed_answer'] = [item['parsed_answer'] for item in regraded]
manual_sample['recomputed_correctness'] = [item['correctness'] for item in regraded]
manual_sample['recomputed_status'] = [item['status'] for item in regraded]
manual_sample['recomputed_reason'] = [item['reason'] for item in regraded]
manual_sample['all_fields_match'] = (
    manual_sample['response'].fillna('') == manual_sample['reparsed_answer'].fillna('')
) & (
    manual_sample['correctness'].fillna(-1) == manual_sample['recomputed_correctness'].fillna(-1)
) & (
    manual_sample['grading_status'].fillna('') == manual_sample['recomputed_status'].fillna('')
) & (
    manual_sample['grading_reason'].fillna('') == manual_sample['recomputed_reason'].fillna('')
)

show_answer(
    f'Checked {len(manual_sample)} usable responses across template types and correctness labels. Any `False` value in `all_fields_match` is a concrete grading mismatch worth debugging immediately.'
)
display(
    manual_sample[
        [
            'record_id', 'template_type', 'question', 'response_raw', 'response', 'reparsed_answer',
            'correctness', 'recomputed_correctness', 'grading_status', 'recomputed_status',
            'grading_reason', 'recomputed_reason', 'all_fields_match'
        ]
    ].reset_index(drop=True)
)


In [ ]:
show_question(6, 'How many responses are ambiguous or unlabeled, what are the main ambiguity reasons, and do those examples look truly ambiguous?')

ambiguous = sampled[~sampled['usable_for_metrics']].copy()
ambiguous_rate = len(ambiguous) / len(sampled) if len(sampled) else np.nan
show_answer(
    f'{len(ambiguous)}/{len(sampled)} sampled responses ({ambiguous_rate:.1%}) are ambiguous or unlabeled. The reason table and sampled examples below help separate true parser ambiguity from labeling bugs.'
)

reason_summary = (
    ambiguous.groupby(['grading_reason', 'template_type'])
    .size()
    .reset_index(name='n_rows')
    .sort_values(['n_rows', 'grading_reason'], ascending=[False, True])
)
display(reason_summary)

example_parts = []
for reason, reason_df in ambiguous.groupby('grading_reason'):
    example_parts.append(reason_df.sample(min(2, len(reason_df)), random_state=AUDIT_SEED + len(reason)))
example_rows = pd.concat(example_parts, ignore_index=True) if example_parts else ambiguous
display(
    example_rows[
['record_id', 'template_type', 'question', 'response_raw', 'response', 'grading_reason', 'gold_answers']
    ].sort_values(['grading_reason', 'record_id']).reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(
    data=ambiguous,
    y='grading_reason',
    hue='template_type',
    hue_order=CTX['template_order'],
    palette=PALETTE,
    ax=ax,
)
finish_ax(ax, 'Ambiguous Responses by Reason', 'Count', 'Grading reason', legend=True, ncol=3)
plt.tight_layout()
plt.show()


In [ ]:
show_question(7, 'What is the model’s accuracy on neutral prompts versus biased prompts, for each bias type and per question?')

acc_by_bias = (
    tuples.groupby('bias_type', as_index=False)
    .agg(
neutral_accuracy=('C_x_y', 'mean'),
biased_accuracy=('C_xprime_yprime', 'mean'),
n_pairs=('draw_idx', 'count'),
    )
)
acc_by_bias['biased_minus_neutral'] = acc_by_bias['biased_accuracy'] - acc_by_bias['neutral_accuracy']

show_answer(
    'Use the summary table for aggregate accuracy and the per-question table for local failures. Negative `biased_minus_neutral` means the bias prompt hurt accuracy relative to the neutral prompt on the exact same question-draw pairs.'
)
display(acc_by_bias.sort_values('biased_minus_neutral'))

plot_df = acc_by_bias.melt(
    id_vars=['bias_type', 'n_pairs', 'biased_minus_neutral'],
    value_vars=['neutral_accuracy', 'biased_accuracy'],
    var_name='prompt_condition',
    value_name='accuracy',
)
plot_df['prompt_condition'] = plot_df['prompt_condition'].map({
    'neutral_accuracy': 'neutral',
    'biased_accuracy': 'biased',
})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=plot_df,
    x='bias_type',
    y='accuracy',
    hue='prompt_condition',
    palette={'neutral': '#73b3ab', 'biased': '#d4651a'},
    ax=ax,
)
finish_ax(ax, 'Neutral vs Biased Accuracy by Bias Type', 'Bias type', 'Accuracy', legend=True, ncol=2)
plt.tight_layout()
plt.show()

per_question = summary.copy()
per_question['biased_minus_neutral'] = per_question['mean_C_xprime'] - per_question['mean_C_x']
display(
    per_question[
['split', 'question_id', 'bias_type', 'question', 'mean_C_x', 'mean_C_xprime', 'biased_minus_neutral', 'n_draws']
    ].sort_values('biased_minus_neutral').reset_index(drop=True)
)


In [ ]:
show_question(8, 'How many probes were trained, on how many usable examples, and what was the performance of each one?')

probe_rows = []
for template_type in CTX['template_order']:
    probe_name = template_to_probe_name(template_type)
    meta = CTX['probe_metadata'].get(probe_name, {})
    train_df = sampled[(sampled['split'] == 'train') & (sampled['template_type'] == template_type)]
    val_df = sampled[(sampled['split'] == 'val') & (sampled['template_type'] == template_type)]
    train_usable = train_df[train_df['usable_for_metrics']]
    val_usable = val_df[val_df['usable_for_metrics']]
    probe_rows.append(
{
    'probe_name': probe_name,
    'template_type': template_type,
    'trained': meta.get('best_layer') is not None,
    'best_layer': meta.get('best_layer'),
    'best_dev_auc': meta.get('best_dev_auc'),
    'train_raw_rows': len(train_df),
    'train_usable_rows': len(train_usable),
    'val_raw_rows': len(val_df),
    'val_usable_rows': len(val_usable),
    'retrain_usable_rows': len(train_usable) + len(val_usable),
    'train_positive_rate': train_usable['correctness'].mean() if len(train_usable) else np.nan,
    'val_positive_rate': val_usable['correctness'].mean() if len(val_usable) else np.nan,
    'saved_selection_models': len(meta.get('saved_selection_models', []) or []),
    'saved_best_model': bool(meta.get('saved_best_model')),
}
    )
probe_df = pd.DataFrame(probe_rows).sort_values(['trained', 'probe_name'], ascending=[False, True]).reset_index(drop=True)

show_answer(
    'This table combines the saved probe metadata with the actual usable train/val sample counts from `sampled_responses.csv`. If a probe was not trained, the class balance columns usually reveal whether the usable subset collapsed to one class.'
)
display(probe_df)


In [ ]:
show_question(9, 'Do probes trained for different bias strategies differ in performance when compared on the same evaluation subset?')

strategy_rows = []
subset_map = {}
for bias_type in CTX['bias_types']:
    probe_name = f'probe_bias_{bias_type}'
    val_usable = sampled[
        (sampled['split'] == 'val')
        & (sampled['template_type'] == bias_type)
        & (sampled['usable_for_metrics'])
    ].copy()
    subset_map[probe_name] = id_tuples(val_usable, ['question_id', 'draw_idx'])
    strategy_rows.append(
        {
            'probe_name': probe_name,
            'bias_type': bias_type,
            'best_dev_auc': CTX['probe_metadata'].get(probe_name, {}).get('best_dev_auc'),
            'best_layer': CTX['probe_metadata'].get(probe_name, {}).get('best_layer'),
            'n_usable_val_rows': len(val_usable),
            'n_usable_val_question_draws': len(subset_map[probe_name]),
            'val_positive_rate': val_usable['correctness'].mean() if len(val_usable) else np.nan,
        }
    )
strategy_df = pd.DataFrame(strategy_rows)

overlap = pd.DataFrame(index=strategy_df['probe_name'], columns=strategy_df['probe_name'], dtype=float)
for left in overlap.index:
    for right in overlap.columns:
        overlap.loc[left, right] = jaccard(subset_map[left], subset_map[right])

show_answer(
    'Direct AUC comparisons are only apples-to-apples when the usable val subsets are identical. Read the Jaccard overlap matrix first: values below 1.0 mean the probes are being compared on different filtered subsets, so any AUC ranking should be treated cautiously.'
)
display(strategy_df)
display(overlap)


In [ ]:
show_question(10, 'Do probe results differ by layer enough to change which layers are worth analyzing in later rounds?')

layer_rows = []
for probe_name, meta in CTX['probe_metadata'].items():
    for layer, auc in (meta.get('auc_per_layer') or {}).items():
        layer_rows.append(
            {
                'probe_name': probe_name,
                'layer': int(layer),
                'auc': np.nan if auc is None else float(auc),
                'best_layer': meta.get('best_layer'),
            }
        )
layer_df = pd.DataFrame(layer_rows).sort_values(['probe_name', 'layer']).reset_index(drop=True)
best_layers = (
    layer_df[['probe_name', 'best_layer']]
    .drop_duplicates()
    .merge(
        pd.DataFrame([
            {'probe_name': name, 'best_dev_auc': meta.get('best_dev_auc')}
            for name, meta in CTX['probe_metadata'].items()
        ]),
        on='probe_name',
        how='left',
    )
)

show_answer(
    'The line plot shows dev AUC by layer for each probe. Large layer-to-layer swings suggest the chosen layer window matters; flat curves suggest later rounds can safely prune the layer search.'
)
display(best_layers.sort_values('probe_name').reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=layer_df.dropna(subset=['auc']),
    x='layer',
    y='auc',
    hue='probe_name',
    palette=PROBE_PALETTE,
    marker='o',
    ax=ax,
)
finish_ax(ax, 'Probe Dev AUC by Layer', 'Layer', 'Dev AUC', legend=True, ncol=2)
plt.tight_layout()
plt.show()

display(layer_df.pivot(index='layer', columns='probe_name', values='auc'))


In [ ]:
show_question(11, 'How many raw rows exist per prompt type, how many survive deduplication, and how many complete question groups remain after requiring all prompt variants?')

raw_type_counts = pd.Series([infer_template_type(row) or 'unknown' for row in CTX['raw_rows']]).value_counts()
dedup_type_counts = pd.Series([infer_template_type(row) or 'unknown' for row in CTX['raw_rows_dedup']]).value_counts()
counts = (
    pd.DataFrame({'raw_rows': raw_type_counts, 'dedup_rows': dedup_type_counts})
    .fillna(0)
    .astype(int)
    .rename_axis('template_type')
    .reset_index()
)

complete_groups_full = len(CTX['groups_all'])
complete_groups_run = len(CTX['groups_for_run'])
show_answer(
    f'After requiring the neutral prompt plus all selected bias variants, {complete_groups_full} complete question groups remain in the source dataset. This run then keeps {complete_groups_run} of them because `max_questions={CTX["run_config"].get("max_questions")}`.'
)
display(counts.sort_values('template_type').reset_index(drop=True))
display(
    pd.DataFrame(
[
    {
        'complete_groups_full_dataset': complete_groups_full,
        'complete_groups_in_this_run': complete_groups_run,
        'max_questions': CTX['run_config'].get('max_questions'),
    }
]
    )
)

counts_long = counts.melt(id_vars='template_type', value_vars=['raw_rows', 'dedup_rows'], var_name='stage', value_name='n_rows')
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=counts_long, x='template_type', y='n_rows', hue='stage', palette={'raw_rows': '#264653', 'dedup_rows': '#73b3ab'}, ax=ax)
finish_ax(ax, 'Raw vs Deduplicated Row Counts', 'Template type', 'Rows', legend=True, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
show_question(12, 'How many questions are dropped because a prompt variant is missing, because gold answers cannot be extracted, or because the question-answer pair is malformed?')

required_types = ['neutral', *CTX['bias_types']]
by_key = {}
for row in CTX['raw_rows_dedup']:
    base = row.get('base', {}) or {}
    key = (
        str(base.get('question', '')).strip(),
        str(base.get('correct_answer', '')).strip(),
        str(base.get('incorrect_answer', '')).strip(),
    )
    by_key.setdefault(key, {})[infer_template_type(row)] = row

reason_rows = []
for key, rows_by_type in by_key.items():
    question, correct_answer, incorrect_answer = key
    neutral_row = rows_by_type.get('neutral') or next(iter(rows_by_type.values()))
    gold_answers = extract_gold_answers_from_base((neutral_row.get('base', {}) or {}))
    missing_templates = [template_type for template_type in required_types if template_type not in rows_by_type]
    missing_prompt_messages = any(
        not isinstance((rows_by_type.get(template_type) or {}).get('prompt'), list)
        or not (rows_by_type.get(template_type) or {}).get('prompt')
        for template_type in required_types
        if template_type in rows_by_type
    )
    malformed_pair = (
        (not question)
        or (not correct_answer)
        or (not incorrect_answer)
        or (normalize_answer(correct_answer) == normalize_answer(incorrect_answer))
    )
    missing_gold_answers = len(gold_answers) == 0
    kept = not malformed_pair and not missing_templates and not missing_gold_answers and not missing_prompt_messages
    primary_reason = 'kept'
    if not kept:
        if malformed_pair:
            primary_reason = 'malformed_pair'
        elif missing_templates:
            primary_reason = 'missing_prompt_variant'
        elif missing_gold_answers:
            primary_reason = 'missing_gold_answers'
        else:
            primary_reason = 'missing_prompt_messages'
    reason_rows.append(
        {
            'question': question,
            'correct_answer': correct_answer,
            'incorrect_answer': incorrect_answer,
            'primary_reason': primary_reason,
            'missing_templates': ', '.join(missing_templates),
            'missing_gold_answers': missing_gold_answers,
            'malformed_pair': malformed_pair,
            'missing_prompt_messages': missing_prompt_messages,
        }
    )
reasons_df = pd.DataFrame(reason_rows)
reason_summary = reasons_df['primary_reason'].value_counts().rename_axis('primary_reason').reset_index(name='n_question_answer_pairs')
max_question_drop = max(0, len(CTX['groups_all']) - len(CTX['groups_for_run']))

show_answer(
    f'The primary-reason table below uses a strict hierarchy: malformed pair, then missing prompt variant, then missing gold answers, then missing prompt messages. Separately, {max_question_drop} otherwise-valid groups are excluded only because the run caps the dataset with `max_questions`.'
)
display(reason_summary)
display(
    reasons_df[reasons_df['primary_reason'] != 'kept'][
        ['question', 'correct_answer', 'incorrect_answer', 'primary_reason', 'missing_templates']
    ].head(20).reset_index(drop=True)
)


In [ ]:
show_question(13, 'Are there duplicate questions or duplicate question-answer pairs that might overweight particular items?')

raw_records = []
for row in CTX['raw_rows']:
    base = row.get('base', {}) or {}
    raw_records.append(
{
    'question': str(base.get('question', '')).strip(),
    'correct_answer': str(base.get('correct_answer', '')).strip(),
    'incorrect_answer': str(base.get('incorrect_answer', '')).strip(),
    'template_type': infer_template_type(row) or 'unknown',
}
    )
raw_df = pd.DataFrame(raw_records)
raw_df['question_norm'] = raw_df['question'].map(lambda text: normalize_answer(text))

duplicate_prompt_rows = (
    raw_df.groupby(['question', 'correct_answer', 'incorrect_answer', 'template_type'])
    .size()
    .reset_index(name='n_raw_rows')
    .query('n_raw_rows > 1')
    .sort_values('n_raw_rows', ascending=False)
)
distinct_pairs = raw_df[['question_norm', 'question', 'correct_answer', 'incorrect_answer']].drop_duplicates()
duplicate_questions = (
    distinct_pairs.groupby(['question_norm', 'question'])
    .size()
    .reset_index(name='n_distinct_answer_pairs')
    .query('n_distinct_answer_pairs > 1')
    .sort_values('n_distinct_answer_pairs', ascending=False)
)
dedup_removed = len(CTX['raw_rows']) - len(CTX['raw_rows_dedup'])

show_answer(
    f'Deduplication removed {dedup_removed} exact prompt-row duplicates from the raw source file. The first table shows repeated prompt rows, and the second shows question texts that appear with multiple distinct answer pairs and could overweight a concept if not handled carefully.'
)
display(duplicate_prompt_rows.head(20).reset_index(drop=True))
display(duplicate_questions.head(20).reset_index(drop=True))


In [ ]:
show_question(14, 'Are train, val, and test truly disjoint at the question level, and are bias types reasonably balanced across those splits?')

spec = (CTX['sampling_manifest'] or {}).get('sampling_spec', {})
split_sets = {
    'train': set(spec.get('train_question_ids', [])),
    'val': set(spec.get('val_question_ids', [])),
    'test': set(spec.get('test_question_ids', [])),
}
overlap_rows = []
for left in ['train', 'val', 'test']:
    for right in ['train', 'val', 'test']:
        if left >= right:
            continue
        overlap_rows.append(
            {
                'split_pair': f'{left} vs {right}',
                'n_overlap_question_ids': len(split_sets[left] & split_sets[right]),
            }
        )
overlap_df = pd.DataFrame(overlap_rows)

balance = (
    sampled.groupby(['split', 'template_type'], as_index=False)
    .agg(
        n_records=('record_id', 'count'),
        n_questions=('question_id', 'nunique'),
        usable_records=('usable_for_metrics', 'sum'),
    )
    .sort_values(['split', 'template_type'])
)

show_answer(
    'The overlap table should be all zeros off-diagonal. The balance table shows question counts, raw record counts, and usable record counts by split and template type so you can catch skew introduced by filtering or ambiguity.'
)
display(overlap_df)
display(balance.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=balance,
    x='split',
    y='usable_records',
    hue='template_type',
    hue_order=CTX['template_order'],
    palette=PALETTE,
    ax=ax,
)
finish_ax(ax, 'Usable Records by Split and Template Type', 'Split', 'Usable records', legend=True, ncol=3)
plt.tight_layout()
plt.show()


In [ ]:
show_question(15, 'For each question, are the correct_answer and incorrect_answer consistent across all prompt variants, with no accidental swaps?')

consistency = (
    sampled.groupby(['split', 'question_id'], as_index=False)
    .agg(
        question=('question', 'first'),
        correct_answer=('correct_answer', 'first'),
        incorrect_answer=('incorrect_answer', 'first'),
        n_correct_answers=('correct_answer', 'nunique'),
        n_incorrect_answers=('incorrect_answer', 'nunique'),
        n_templates=('template_type', 'nunique'),
    )
)
inconsistent_answers = consistency[
    (consistency['n_correct_answers'] > 1) | (consistency['n_incorrect_answers'] > 1)
].copy()

def prompt_alignment_ok(row):
    prompt_text = str(row['prompt_text'])
    if row['template_type'] == 'neutral':
        return prompt_text.strip() == str(row['question']).strip()
    if row['template_type'] == 'incorrect_suggestion':
        return str(row['incorrect_answer']) in prompt_text
    if row['template_type'] in {'doubt_correct', 'suggest_correct'}:
        return str(row['correct_answer']) in prompt_text
    return True

prompt_checks = sampled[
    ['record_id', 'split', 'question_id', 'template_type', 'question', 'prompt_text', 'correct_answer', 'incorrect_answer']
].drop_duplicates().copy()
prompt_checks['prompt_alignment_ok'] = prompt_checks.apply(prompt_alignment_ok, axis=1)
prompt_failures = prompt_checks[~prompt_checks['prompt_alignment_ok']].copy()

show_answer(
    f'{len(inconsistent_answers)} question ids have inconsistent answer fields across prompt variants, and {len(prompt_failures)} prompt texts fail a template-specific answer-insertion check. Any non-zero result here is a data construction bug, not a model behavior issue.'
)
display(inconsistent_answers.reset_index(drop=True))
display(prompt_failures.reset_index(drop=True))


In [ ]:
show_question(16, 'Are ambiguous or no-label cases concentrated in certain answer formats, question types, or bias templates?')

amb = sampled[~sampled['usable_for_metrics']].copy()
amb['question_type'] = amb['question'].map(question_type_bucket)
amb['parsed_format'] = amb['response'].map(answer_format_bucket)
sampled_with_buckets = sampled.copy()
sampled_with_buckets['question_type'] = sampled_with_buckets['question'].map(question_type_bucket)

ambiguous_rate_by_template = (
    sampled.groupby('template_type')['usable_for_metrics']
    .agg(lambda values: 1 - float(values.mean()))
    .rename('ambiguous_rate')
    .reset_index()
    .sort_values('ambiguous_rate', ascending=False)
)
ambiguous_rate_by_qtype = (
    sampled_with_buckets.groupby('question_type')
    .agg(total_rows=('record_id', 'count'), ambiguous_rows=('usable_for_metrics', lambda values: int((~values).sum())))
    .reset_index()
)
ambiguous_rate_by_qtype['ambiguous_rate'] = ambiguous_rate_by_qtype['ambiguous_rows'] / ambiguous_rate_by_qtype['total_rows']
ambiguous_rate_by_qtype = ambiguous_rate_by_qtype.sort_values(['ambiguous_rate', 'total_rows'], ascending=[False, False])

reason_format = (
    amb.groupby(['grading_reason', 'parsed_format'])
    .size()
    .reset_index(name='n_rows')
    .sort_values('n_rows', ascending=False)
)

show_answer(
    'Read the template-level rates first, then the question-type and answer-format tables. Concentration in one template or one format usually points to a parser issue or a brittle prompt template rather than broad model uncertainty.'
)
display(ambiguous_rate_by_template)
display(ambiguous_rate_by_qtype.head(15).reset_index(drop=True))
display(reason_format.head(20).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=ambiguous_rate_by_template,
    x='template_type',
    y='ambiguous_rate',
    hue='template_type',
    palette={template_type: PALETTE.get(template_type, '#8d99ae') for template_type in ambiguous_rate_by_template['template_type']},
    dodge=False,
    legend=False,
    ax=ax,
)
finish_ax(ax, 'Ambiguous Rate by Template Type', 'Template type', 'Ambiguous rate')
plt.tight_layout()
plt.show()


In [ ]:
show_question(17, 'When one side of a neutral/bias pair is ambiguous and the other is usable, how much data do we lose from pairwise analyses, and could that distort the estimated bias effect?')

neutral = sampled[sampled['template_type'] == 'neutral'][['split', 'question_id', 'draw_idx', 'usable_for_metrics']].rename(columns={'usable_for_metrics': 'neutral_usable'})
pair_parts = []
for bias_type in CTX['bias_types']:
    bias = sampled[sampled['template_type'] == bias_type][['split', 'question_id', 'draw_idx', 'usable_for_metrics']].rename(columns={'usable_for_metrics': 'bias_usable'})
    merged = neutral.merge(bias, on=['split', 'question_id', 'draw_idx'], how='outer')
    merged['bias_type'] = bias_type
    merged['neutral_usable'] = merged['neutral_usable'].fillna(False)
    merged['bias_usable'] = merged['bias_usable'].fillna(False)
    merged['pair_status'] = np.select(
[
    merged['neutral_usable'] & merged['bias_usable'],
    merged['neutral_usable'] & ~merged['bias_usable'],
    ~merged['neutral_usable'] & merged['bias_usable'],
],
['both_usable', 'neutral_only_usable', 'bias_only_usable'],
default='neither_usable',
    )
    pair_parts.append(merged)
pair_df = pd.concat(pair_parts, ignore_index=True)

loss_summary = (
    pair_df.groupby(['bias_type', 'pair_status'])
    .size()
    .reset_index(name='n_pairs')
    .sort_values(['bias_type', 'pair_status'])
)
loss_rates = (
    pair_df.assign(lost=lambda df: df['pair_status'] != 'both_usable')
    .groupby('bias_type')['lost']
    .mean()
    .rename('loss_rate')
    .reset_index()
    .sort_values('loss_rate', ascending=False)
)

show_answer(
    'Pairwise accuracy analyses only keep `both_usable` rows. The loss-rate table below shows how much data disappears when one side of a neutral/bias pair is ambiguous, which is exactly where pairwise estimates can become biased by filtering.'
)
display(loss_summary.reset_index(drop=True))
display(loss_rates.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=loss_summary,
    x='bias_type',
    y='n_pairs',
    hue='pair_status',
    palette={
'both_usable': '#264653',
'neutral_only_usable': '#73b3ab',
'bias_only_usable': '#d4651a',
'neither_usable': '#bcb8b1',
    },
    ax=ax,
)
finish_ax(ax, 'Pairwise Retention vs Loss', 'Bias type', 'Question-draw pairs', legend=True, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
show_question(18, 'Are the largest bias effects coming from already-hard questions, or do they also appear on questions the model usually gets right?')

effect_df = summary.copy()
effect_df['bias_effect'] = effect_df['mean_C_xprime'] - effect_df['mean_C_x']
effect_df['neutral_hardness'] = 1 - effect_df['mean_C_x']

show_answer(
    'Look for large negative bias effects near neutral hardness 0.0 to find questions the model usually gets right until the bias text is added. Large negative effects only at high hardness suggest the bias mainly hurts already-fragile items.'
)
display(
    effect_df[
['split', 'question_id', 'bias_type', 'question', 'mean_C_x', 'mean_C_xprime', 'neutral_hardness', 'bias_effect']
    ].sort_values('bias_effect').reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    data=effect_df,
    x='neutral_hardness',
    y='bias_effect',
    hue='bias_type',
    palette=PALETTE,
    s=90,
    ax=ax,
)
ax.axhline(0, color='black', linestyle='--', linewidth=1.2)
finish_ax(ax, 'Bias Effect vs Neutral Hardness', 'Neutral hardness (1 - neutral accuracy)', 'Biased - neutral accuracy', legend=True, ncol=3)
plt.tight_layout()
plt.show()


In [ ]:
show_question(19, 'Is the observed bias effect stable across multiple draws, or is it driven by a small number of noisy samples?')

draw_df = tuples.copy()
draw_df['bias_effect'] = draw_df['C_xprime_yprime'] - draw_df['C_x_y']
agg_by_draw = (
    draw_df.groupby(['bias_type', 'draw_idx'], as_index=False)
    .agg(mean_bias_effect=('bias_effect', 'mean'), n_pairs=('question_id', 'count'))
    .sort_values(['bias_type', 'draw_idx'])
)

pivot = draw_df.pivot_table(index=['split', 'question_id', 'bias_type'], columns='draw_idx', values='bias_effect', aggfunc='first').reset_index()
draw_cols = [column for column in pivot.columns if isinstance(column, (int, np.integer))]
if len(draw_cols) >= 2:
    pivot['same_sign_across_draws'] = np.sign(pivot[draw_cols[0]].fillna(0)) == np.sign(pivot[draw_cols[1]].fillna(0))
    sign_agreement = float(pivot['same_sign_across_draws'].mean())
else:
    pivot['same_sign_across_draws'] = np.nan
    sign_agreement = np.nan

rng = np.random.default_rng(AUDIT_SEED + 19)
bootstrap_rows = []
for bias_type, bias_df in draw_df.groupby('bias_type'):
    values = bias_df['bias_effect'].to_numpy(dtype=float)
    if len(values) == 0:
        continue
    boot = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(500)])
    bootstrap_rows.append(
        {
            'bias_type': bias_type,
            'mean_bias_effect': float(values.mean()),
            'bootstrap_ci_low': float(np.quantile(boot, 0.025)),
            'bootstrap_ci_high': float(np.quantile(boot, 0.975)),
        }
    )
bootstrap_df = pd.DataFrame(bootstrap_rows)

show_answer(
    f'With only {draw_df["draw_idx"].nunique()} draws, the strongest stability checks are draw-level agreement and per-question sign consistency. Current sign agreement across available draw pairs is {sign_agreement:.1%}.'
)
display(agg_by_draw)
display(bootstrap_df)
display(pivot.head(20).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(9, 5))
sns.stripplot(
    data=draw_df,
    x='bias_type',
    y='bias_effect',
    hue='draw_idx',
    palette={0: '#73b3ab', 1: '#d4651a'},
    dodge=True,
    ax=ax,
)
finish_ax(ax, 'Bias Effect by Draw', 'Bias type', 'Biased - neutral accuracy', legend=True, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
show_question(20, 'When comparing probes, strategies, or layers, am I always comparing them on the same usable set of examples rather than different filtered subsets?')

notes = pd.DataFrame(
    [
        {
            'analysis': 'Neutral vs biased accuracy',
            'same_usable_set': 'Yes',
            'subset_definition': 'Rows in final_tuples.csv, which already require both neutral and biased sides of the same question-draw pair to be usable.',
        },
        {
            'analysis': 'Per-question accuracy summaries',
            'same_usable_set': 'Yes',
            'subset_definition': 'Aggregated from final_tuples.csv, so they inherit the same paired usable subset.',
        },
        {
            'analysis': 'Layer comparison within one probe',
            'same_usable_set': 'Yes',
            'subset_definition': 'The pipeline uses the same template-specific usable train/val records for every layer inside a single probe.',
        },
        {
            'analysis': 'Probe comparison across strategies',
            'same_usable_set': 'Only if overlap = 1.0',
            'subset_definition': 'Each strategy uses its own usable val subset after ambiguity filtering, so cross-strategy AUC comparisons can be apples-to-oranges.',
        },
    ]
)

probe_subset_map = {}
for template_type in CTX['template_order']:
    probe_name = template_to_probe_name(template_type)
    val_usable = sampled[
        (sampled['split'] == 'val')
        & (sampled['template_type'] == template_type)
        & (sampled['usable_for_metrics'])
    ]
    probe_subset_map[probe_name] = id_tuples(val_usable, ['question_id', 'draw_idx'])

overlap = pd.DataFrame(index=probe_subset_map.keys(), columns=probe_subset_map.keys(), dtype=float)
for left in overlap.index:
    for right in overlap.columns:
        overlap.loc[left, right] = jaccard(probe_subset_map[left], probe_subset_map[right])

subset_sizes = pd.DataFrame(
    [
        {'subset_name': 'final_tuples_pairs', 'n_ids': len(id_tuples(tuples, ['split', 'question_id', 'bias_type', 'draw_idx']))},
        *[
            {'subset_name': probe_name, 'n_ids': len(ids)}
            for probe_name, ids in probe_subset_map.items()
        ],
    ]
)

show_answer(
    'Use the notes table as the rule of thumb for apples-to-apples comparisons. For probes, the overlap matrix below is the concrete check: off-diagonal values below 1.0 mean different strategies are not being evaluated on the exact same usable examples.'
)
display(notes)
display(subset_sizes)
display(overlap)
